In [ ]:
from google.colab import drive
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
drive.mount("/content/drive", force_remount=True)


In [ ]:
# @title Saved-model bank manifest and loader
# =========================
# WHAT THIS CELL CONTAINS
# =========================
# Saved bank convention:
#   {save_root}/{run_group}/{dataset_name}/{model_name}/seed_{seed:02d}/
#       meta.json
#       done.json
#       best_analysis.msgpack
#       last_analysis.msgpack
#       best_metrics.json
#       history.csv
#
# Exact 6 parameters needed to load a saved model:
#   1) save_root
#   2) run_group
#   3) dataset_name      in {"cifar10", "cifar100", "imagenet100"}
#   4) model_name        in {"resnet18", "small_vit"}
#   5) seed              int, usually 0..9
#   6) checkpoint_tag    in {"best", "last"}
#
# Main loader:
#   bundle = load_rep_bundle(save_root, run_group, dataset_name, model_name, seed, checkpoint_tag="best")
#
# Returned bundle keys:
#   "run_dir"
#   "meta"
#   "model"
#   "variables"
#   "step"
#   "layer_names"
#   "apply_logits"      : function x -> logits
#   "get_activations"   : function x -> (logits, acts_dict)
#   "get_layer"         : function x, layer_name -> activation
#   "jvp_logits"        : function x, v -> (primal, tangent)
#   "jvp_layer"         : function x, v, layer_name -> (primal, tangent)
#
# Important:
# - This loader expects the "fixed names" model version where intermediate activations are:
#     ResNet18: act_stem, act_block1..act_block8, act_pre_logits
#     SmallViT: act_patch_grid, act_tokens_in, act_encoderblock_1..8, act_pre_logits
# - The saved checkpoints are lightweight analysis payloads, not full optimizer checkpoints.

import re
import json
from pathlib import Path
from typing import Any, Dict

import numpy as np
import jax
import jax.numpy as jnp
from jax import random
import flax
import flax.linen as nn
from flax import serialization

# -------------------------
# Default bank location
# -------------------------
DEFAULT_SAVE_ROOT = "/content/drive/MyDrive/representation_bank"
DEFAULT_RUN_GROUP = "repbank_clean_v1"

# -------------------------
# Dataset manifest
# -------------------------
DATASET_CFG = {
    "cifar10": {
        "num_classes": 10,
        "image_size": 32,
        "mean": (0.4914, 0.4822, 0.4465),
        "std":  (0.2470, 0.2435, 0.2616),
    },
    "cifar100": {
        "num_classes": 100,
        "image_size": 32,
        "mean": (0.5071, 0.4867, 0.4408),
        "std":  (0.2675, 0.2565, 0.2761),
    },
    "imagenet100": {
        "num_classes": 100,
        "image_size": 224,
        "mean": (0.485, 0.456, 0.406),
        "std":  (0.229, 0.224, 0.225),
    },
}

# -------------------------
# Model manifest
# -------------------------
MODEL_MANIFEST = {
    "resnet18": {
        "family": "cnn",
        "notes": "CIFAR-style stem for 32x32 inputs; ImageNet-style stem for 224x224 inputs.",
        "intermediate_layers_expected": [
            "act_stem",
            "act_block1", "act_block2", "act_block3", "act_block4",
            "act_block5", "act_block6", "act_block7", "act_block8",
            "act_pre_logits",
        ],
    },
    "small_vit": {
        "family": "vit",
        "notes": "Patch size 4 on 32x32 inputs, patch size 16 on 224x224 inputs; depth 8.",
        "intermediate_layers_expected": [
            "act_patch_grid",
            "act_tokens_in",
            "act_encoderblock_1", "act_encoderblock_2", "act_encoderblock_3", "act_encoderblock_4",
            "act_encoderblock_5", "act_encoderblock_6", "act_encoderblock_7", "act_encoderblock_8",
            "act_pre_logits",
        ],
    },
}

def natural_key(s: str):
    return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", s)]

def run_dir(save_root, run_group, dataset_name, model_name, seed):
    return Path(save_root) / run_group / dataset_name / model_name / f"seed_{seed:02d}"

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# MODEL DEFINITIONS
# =========================

class ResidualBlock(nn.Module):
    features: int
    stride: int = 1

    @nn.compact
    def __call__(self, x, train: bool):
        residual = x

        y = nn.Conv(
            self.features, (3, 3),
            strides=(self.stride, self.stride),
            padding="SAME",
            use_bias=False,
        )(x)
        y = nn.BatchNorm(
            use_running_average=not train,
            momentum=0.9,
            epsilon=1e-5,
        )(y)
        y = nn.relu(y)

        y = nn.Conv(
            self.features, (3, 3),
            strides=(1, 1),
            padding="SAME",
            use_bias=False,
        )(y)
        y = nn.BatchNorm(
            use_running_average=not train,
            momentum=0.9,
            epsilon=1e-5,
            scale_init=nn.initializers.zeros,
        )(y)

        if residual.shape != y.shape:
            residual = nn.Conv(
                self.features, (1, 1),
                strides=(self.stride, self.stride),
                use_bias=False,
            )(residual)
            residual = nn.BatchNorm(
                use_running_average=not train,
                momentum=0.9,
                epsilon=1e-5,
            )(residual)

        return nn.relu(residual + y)

class ResNet18(nn.Module):
    num_classes: int
    image_size: int

    @nn.compact
    def __call__(self, x, train: bool):
        small_input = self.image_size <= 64

        if small_input:
            x = nn.Conv(
                64, (3, 3),
                strides=(1, 1),
                padding="SAME",
                use_bias=False,
                name="stem_conv",
            )(x)
        else:
            x = nn.Conv(
                64, (7, 7),
                strides=(2, 2),
                padding="SAME",
                use_bias=False,
                name="stem_conv",
            )(x)

        x = nn.BatchNorm(
            use_running_average=not train,
            momentum=0.9,
            epsilon=1e-5,
            name="stem_bn",
        )(x)
        x = nn.relu(x)
        self.sow("intermediates", "act_stem", x)

        if not small_input:
            x = nn.max_pool(x, window_shape=(3, 3), strides=(2, 2), padding="SAME")

        block_specs = [
            (64, 1), (64, 1),
            (128, 2), (128, 1),
            (256, 2), (256, 1),
            (512, 2), (512, 1),
        ]

        for i, (features, stride) in enumerate(block_specs, start=1):
            x = ResidualBlock(features=features, stride=stride, name=f"block{i}")(x, train=train)
            self.sow("intermediates", f"act_block{i}", x)

        x = jnp.mean(x, axis=(1, 2))
        self.sow("intermediates", "act_pre_logits", x)
        logits = nn.Dense(self.num_classes, name="head")(x)
        return logits

class DropPath(nn.Module):
    rate: float = 0.0

    @nn.compact
    def __call__(self, x, train: bool):
        if (not train) or self.rate == 0.0:
            return x
        keep_prob = 1.0 - self.rate
        mask_shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = jax.random.bernoulli(self.make_rng("drop_path"), p=keep_prob, shape=mask_shape)
        return x * mask.astype(x.dtype) / keep_prob

class MLP(nn.Module):
    hidden_dim: int
    out_dim: int
    drop_rate: float = 0.0

    @nn.compact
    def __call__(self, x, train: bool):
        x = nn.Dense(self.hidden_dim)(x)
        x = nn.gelu(x)
        x = nn.Dropout(self.drop_rate)(x, deterministic=not train)
        x = nn.Dense(self.out_dim)(x)
        x = nn.Dropout(self.drop_rate)(x, deterministic=not train)
        return x

class EncoderBlock(nn.Module):
    dim: int
    num_heads: int
    mlp_ratio: float = 4.0
    drop_rate: float = 0.0
    drop_path_rate: float = 0.0

    @nn.compact
    def __call__(self, x, train: bool):
        y = nn.LayerNorm()(x)
        y = nn.MultiHeadDotProductAttention(
            num_heads=self.num_heads,
            qkv_features=self.dim,
            out_features=self.dim,
            dropout_rate=self.drop_rate,
        )(y, y, deterministic=not train)
        y = DropPath(self.drop_path_rate)(y, train=train)
        x = x + y

        y = nn.LayerNorm()(x)
        y = MLP(
            hidden_dim=int(self.dim * self.mlp_ratio),
            out_dim=self.dim,
            drop_rate=self.drop_rate,
        )(y, train=train)
        y = DropPath(self.drop_path_rate)(y, train=train)
        x = x + y
        return x

class SmallViT(nn.Module):
    num_classes: int
    image_size: int
    patch_size: int
    embed_dim: int = 256
    depth: int = 8
    num_heads: int = 8
    mlp_ratio: float = 4.0
    drop_rate: float = 0.0
    drop_path_rate: float = 0.10

    @nn.compact
    def __call__(self, x, train: bool):
        x = nn.Conv(
            features=self.embed_dim,
            kernel_size=(self.patch_size, self.patch_size),
            strides=(self.patch_size, self.patch_size),
            padding="VALID",
            name="patch_embed",
        )(x)
        self.sow("intermediates", "act_patch_grid", x)

        b, h, w, c = x.shape
        x = x.reshape((b, h * w, c))
        self.sow("intermediates", "act_tokens_in", x)

        cls_token = self.param("cls_token", nn.initializers.zeros, (1, 1, c))
        pos_embed = self.param(
            "pos_embedding",
            nn.initializers.normal(stddev=0.02),
            (1, x.shape[1] + 1, c),
        )

        cls_tokens = jnp.tile(cls_token, (b, 1, 1))
        x = jnp.concatenate([cls_tokens, x], axis=1)
        x = x + pos_embed
        x = nn.Dropout(self.drop_rate)(x, deterministic=not train)

        for i in range(self.depth):
            dpr = self.drop_path_rate * (i / max(1, self.depth - 1))
            x = EncoderBlock(
                dim=self.embed_dim,
                num_heads=self.num_heads,
                mlp_ratio=self.mlp_ratio,
                drop_rate=self.drop_rate,
                drop_path_rate=dpr,
                name=f"encoderblock_{i+1}",
            )(x, train=train)
            self.sow("intermediates", f"act_encoderblock_{i+1}", x)

        x = nn.LayerNorm(name="encoder_norm")(x)
        self.sow("intermediates", "act_pre_logits", x[:, 0])
        logits = nn.Dense(self.num_classes, name="head")(x[:, 0])
        return logits

def build_model(model_name: str, dataset_name: str):
    num_classes = DATASET_CFG[dataset_name]["num_classes"]
    image_size = DATASET_CFG[dataset_name]["image_size"]

    if model_name == "resnet18":
        model = ResNet18(num_classes=num_classes, image_size=image_size)
        model_cfg = {
            "model_name": model_name,
            "num_classes": num_classes,
            "image_size": image_size,
        }
        return model, model_cfg

    if model_name == "small_vit":
        patch_size = 4 if image_size <= 32 else 16
        embed_dim = 256 if image_size <= 32 else 384
        num_heads = 8 if image_size <= 32 else 6

        model = SmallViT(
            num_classes=num_classes,
            image_size=image_size,
            patch_size=patch_size,
            embed_dim=embed_dim,
            depth=8,
            num_heads=num_heads,
            mlp_ratio=4.0,
            drop_rate=0.0,
            drop_path_rate=0.10,
        )
        model_cfg = {
            "model_name": model_name,
            "num_classes": num_classes,
            "image_size": image_size,
            "patch_size": patch_size,
            "embed_dim": embed_dim,
            "depth": 8,
            "num_heads": num_heads,
            "mlp_ratio": 4.0,
            "drop_rate": 0.0,
            "drop_path_rate": 0.10,
        }
        return model, model_cfg

    raise ValueError(f"Unknown model_name: {model_name}")

def get_layer_names(model, dataset_name):
    image_size = DATASET_CFG[dataset_name]["image_size"]
    dummy = jnp.zeros((1, image_size, image_size, 3), dtype=jnp.float32)
    key = random.PRNGKey(0)
    vars0 = model.init({"params": key, "dropout": key, "drop_path": key}, dummy, train=False)
    _, mut = model.apply(vars0, dummy, train=False, mutable=["intermediates"])
    return sorted(list(mut["intermediates"].keys()), key=natural_key)

# =========================
# LOADER
# =========================

def load_rep_bundle(save_root, run_group, dataset_name, model_name, seed, checkpoint_tag="best"):
    """
    Exact 6-parameter loader.

    Parameters
    ----------
    save_root : str or Path
    run_group : str
    dataset_name : {"cifar10", "cifar100", "imagenet100"}
    model_name : {"resnet18", "small_vit"}
    seed : int
    checkpoint_tag : {"best", "last"}
    """
    if checkpoint_tag not in ("best", "last"):
        raise ValueError("checkpoint_tag must be 'best' or 'last'")

    rdir = run_dir(save_root, run_group, dataset_name, model_name, seed)
    meta_path = rdir / "meta.json"
    ckpt_path = rdir / f"{checkpoint_tag}_analysis.msgpack"

    if not meta_path.exists():
        raise FileNotFoundError(f"Missing metadata file: {meta_path}")
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Missing checkpoint file: {ckpt_path}")

    meta = load_json(meta_path)
    model, model_cfg = build_model(model_name, dataset_name)

    image_size = meta["dataset_cfg"]["image_size"]
    dummy = jnp.zeros((1, image_size, image_size, 3), dtype=jnp.float32)
    key = random.PRNGKey(0)

    init_vars = model.init(
        {"params": key, "dropout": key, "drop_path": key},
        dummy,
        train=False,
    )

    template = {
        "params": init_vars["params"],
        "batch_stats": init_vars.get("batch_stats", {}),
        "step": np.asarray(0, dtype=np.int32),
    }

    with open(ckpt_path, "rb") as f:
        payload = serialization.from_bytes(template, f.read())

    variables = {"params": payload["params"]}
    if payload["batch_stats"]:
        variables["batch_stats"] = payload["batch_stats"]

    layer_names = meta.get("layer_names", get_layer_names(model, dataset_name))

    def apply_logits(x, train=False):
        return model.apply(variables, x, train=train)

    def get_activations(x, train=False):
        logits, mut = model.apply(variables, x, train=train, mutable=["intermediates"])
        acts = {k: v[0] for k, v in mut["intermediates"].items()}
        return logits, acts

    def get_layer(x, layer_name, train=False):
        _, acts = get_activations(x, train=train)
        return acts[layer_name]

    def jvp_logits(x, v):
        f = lambda z: model.apply(variables, z, train=False)
        return jax.jvp(f, (x,), (v,))

    def jvp_layer(x, v, layer_name):
        def f(z):
            _, acts = get_activations(z, train=False)
            return acts[layer_name]
        return jax.jvp(f, (x,), (v,))

    bundle = {
        "run_dir": str(rdir),
        "meta": meta,
        "model_cfg": model_cfg,
        "model": model,
        "variables": variables,
        "step": int(payload["step"]),
        "layer_names": layer_names,
        "apply_logits": apply_logits,
        "get_activations": get_activations,
        "get_layer": get_layer,
        "jvp_logits": jvp_logits,
        "jvp_layer": jvp_layer,
    }
    return bundle

# =========================
# OPTIONAL SUMMARY HELPERS
# =========================

def describe_saved_run(save_root=DEFAULT_SAVE_ROOT, run_group=DEFAULT_RUN_GROUP,
                       dataset_name="cifar10", model_name="resnet18", seed=0):
    rdir = run_dir(save_root, run_group, dataset_name, model_name, seed)
    meta = load_json(rdir / "meta.json")
    out = {
        "run_dir": str(rdir),
        "dataset_name": dataset_name,
        "model_name": model_name,
        "seed": seed,
        "available_files": sorted([p.name for p in rdir.iterdir()]) if rdir.exists() else [],
        "image_size": meta["dataset_cfg"]["image_size"],
        "num_classes": meta["dataset_cfg"]["num_classes"],
        "layer_names": meta["layer_names"],
        "schedule_keys": list(meta["schedule"].keys()) if "schedule" in meta else [],
    }
    return out

# Example load:
# bundle = load_rep_bundle(
#     save_root=DEFAULT_SAVE_ROOT,
#     run_group=DEFAULT_RUN_GROUP,
#     dataset_name="cifar10",
#     model_name="resnet18",
#     seed=0,
#     checkpoint_tag="best",
# )
#
# x = jnp.zeros((4, 32, 32, 3), dtype=jnp.float32)
# v = jnp.ones_like(x) * 1e-3
# logits, acts = bundle["get_activations"](x)
# print(bundle["layer_names"])
# print(logits.shape, acts["act_pre_logits"].shape)
# primal, tangent = bundle["jvp_logits"](x, v)
# print(primal.shape, tangent.shape)

print("Handoff cell loaded.")
print("Default save_root:", DEFAULT_SAVE_ROOT)
print("Default run_group:", DEFAULT_RUN_GROUP)
print("Datasets:", list(DATASET_CFG.keys()))
print("Models:", list(MODEL_MANIFEST.keys()))
print("Loader signature: load_rep_bundle(save_root, run_group, dataset_name, model_name, seed, checkpoint_tag='best')")

In [ ]:
# @title 1. Setup, imports, locked config

!pip install -q flax tensorflow tensorflow-datasets scipy pandas seaborn scikit-learn

import os
import gc
import re
import json
import itertools
import numpy as np
import pandas as pd
import scipy.stats as stats
import scipy.ndimage as ndi

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from functools import partial
from tqdm.notebook import tqdm

import jax
import jax.numpy as jnp
import tensorflow as tf
import tensorflow_datasets as tfds

from sklearn.metrics import roc_auc_score

# ---------------------------------------------------
# REQUIRED: run your handoff / loader cell first
# ---------------------------------------------------
required_globals = [
    "DEFAULT_SAVE_ROOT",
    "DEFAULT_RUN_GROUP",
    "DATASET_CFG",
    "load_rep_bundle",
]
missing = [x for x in required_globals if x not in globals()]
if len(missing) > 0:
    raise RuntimeError(
        "You need to run the handoff loader cell first. Missing: " + ", ".join(missing)
    )

try:
    import jax.tools.colab_tpu
    jax.tools.colab_tpu.setup_tpu()
except Exception:
    pass

print("JAX backend:", jax.default_backend())
print("Local devices:", jax.local_device_count())

sns.set_context("talk")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300

# ---------------------------------------------------
# LOCKED CONFIG
# ---------------------------------------------------
GLOBAL_SEED = 0
rng = np.random.default_rng(GLOBAL_SEED)

BANK_DATASET = "cifar10"   # change to "cifar100" later if needed
CHECKPOINT_TAG = "best"

BANK_MODELS = ["resnet18", "small_vit"]
BANK_SEEDS = list(range(10))

REP_LAYER = "act_pre_logits"

# broad boundary-safe smooth geometric family
MODE_LIST = [
    (1, 1), (1, 2), (2, 1), (2, 2),
    (1, 3), (3, 1), (2, 3), (3, 2),
    (3, 3), (1, 4), (4, 1), (2, 4),
]
N_SCALAR_MODES = len(MODE_LIST)
N_BROAD_FLOWS = 2 * N_SCALAR_MODES   # 24 total

# learned family dimension
K_FAMILY = 32

# finite-difference coefficient for tangent estimation wrt flow basis
FD_EPS = 0.5

# family learning / geometry splits
N_FAM_LEARN = 500
N_GEOM = 2000

# benchmark image selection
N_BENCH_CANDIDATES = 256
N_BENCH_IMAGES = 128
MIN_CLEAN_CORRECT_TOTAL = 12
MIN_CLEAN_CORRECT_PER_GROUP_ALLMODELS = 4

# discovery/test splits over models
N_SPLITS = 10
DISC_PER_GROUP = 5
TEST_PER_GROUP = 5

# class-conditional probe design
PROBES_PER_SIDE = 2           # 2 favoring group A, 2 favoring group B
N_RANDOM_CAND = 64
TOP_POOLED_EIGS = 8

# finite perturbation amplitudes in RMS flow pixels
FLOW_RMS_AMPS = np.array([0.5, 1.0], dtype=np.float32)

# batching
FEATURE_BATCH_SIZE = 256
GRAM_BATCH_SIZE_TOTAL = 32
LOGIT_BATCH_SIZE = 256

# held-out image/group validity
MIN_GROUP_MODELS_PER_IMAGE = 2

# bootstrap
N_BOOT = 500

# ---------------------------------------------------
# Debug-first toggle
# ---------------------------------------------------
DEBUG = False
if DEBUG:
    N_FAM_LEARN = 128
    N_GEOM = 256
    N_BENCH_CANDIDATES = 64
    N_BENCH_IMAGES = 24
    MIN_CLEAN_CORRECT_TOTAL = 8
    MIN_CLEAN_CORRECT_PER_GROUP_ALLMODELS = 2
    N_SPLITS = 4
    N_BOOT = 100

# save paths
ROOT_DIR = "/content/drive/MyDrive/local_sensitivity_class_conditional_diag_probes"
RUN_TAG = f"class_conditional_diag_probes_{BANK_DATASET}_k{K_FAMILY}"
RUN_DIR = os.path.join(ROOT_DIR, RUN_TAG)
FIG_DIR = os.path.join(RUN_DIR, "figures")
ARRAY_DIR = os.path.join(RUN_DIR, "arrays")
TABLE_DIR = os.path.join(RUN_DIR, "tables")

for d in [RUN_DIR, FIG_DIR, ARRAY_DIR, TABLE_DIR]:
    os.makedirs(d, exist_ok=True)

print("Saving to:")
print(" ", RUN_DIR)


In [ ]:
# @title 2. Load dataset, normalize, deterministic splits

def load_tfds_numpy(dataset_name, split):
    ds = tfds.load(dataset_name, split=split, batch_size=-1, as_supervised=True)
    x, y = tfds.as_numpy(ds)
    x = x.astype(np.float32) / 255.0
    y = y.astype(np.int64)

    mean = np.array(DATASET_CFG[dataset_name]["mean"], dtype=np.float32)
    std = np.array(DATASET_CFG[dataset_name]["std"], dtype=np.float32)

    x = (x - mean[None, None, None, :]) / std[None, None, None, :]
    return x, y

def normalize_from_pixel(x_pix, dataset_name):
    mean = np.array(DATASET_CFG[dataset_name]["mean"], dtype=np.float32)
    std = np.array(DATASET_CFG[dataset_name]["std"], dtype=np.float32)
    return (x_pix - mean[None, None, :]) / std[None, None, :]

def denormalize_to_pixel(x_norm, dataset_name):
    mean = np.array(DATASET_CFG[dataset_name]["mean"], dtype=np.float32)
    std = np.array(DATASET_CFG[dataset_name]["std"], dtype=np.float32)
    return np.clip(x_norm * std[None, None, :] + mean[None, None, :], 0.0, 1.0)

train_cache = os.path.join(ARRAY_DIR, f"{BANK_DATASET}_train_full.npz")
test_cache = os.path.join(ARRAY_DIR, f"{BANK_DATASET}_test_full.npz")

if os.path.exists(train_cache) and os.path.exists(test_cache):
    tr = np.load(train_cache)
    te = np.load(test_cache)
    X_train_full, y_train_full = tr["X"], tr["y"]
    X_test_full, y_test_full = te["X"], te["y"]
else:
    X_train_full, y_train_full = load_tfds_numpy(BANK_DATASET, "train")
    X_test_full, y_test_full = load_tfds_numpy(BANK_DATASET, "test")
    np.savez_compressed(train_cache, X=X_train_full, y=y_train_full)
    np.savez_compressed(test_cache, X=X_test_full, y=y_test_full)

print("Train:", X_train_full.shape, y_train_full.shape)
print("Test :", X_test_full.shape, y_test_full.shape)

train_perm = np.random.default_rng(GLOBAL_SEED).permutation(len(X_train_full))
test_perm = np.random.default_rng(GLOBAL_SEED + 1).permutation(len(X_test_full))

famlearn_idx = train_perm[:N_FAM_LEARN]
geom_idx = train_perm[N_FAM_LEARN:N_FAM_LEARN + N_GEOM]
cand_idx = test_perm[:N_BENCH_CANDIDATES]

X_famlearn = X_train_full[famlearn_idx]
y_famlearn = y_train_full[famlearn_idx]

X_geom = X_train_full[geom_idx]
y_geom = y_train_full[geom_idx]

X_cand = X_test_full[cand_idx]
y_cand = y_test_full[cand_idx]

N_CLASSES = int(np.max(y_train_full)) + 1

print("Family-learning split:", X_famlearn.shape)
print("Geometry split:", X_geom.shape)
print("Candidate benchmark pool:", X_cand.shape)
print("Num classes:", N_CLASSES)

np.savez_compressed(
    os.path.join(ARRAY_DIR, "initial_split_indices.npz"),
    famlearn_idx=famlearn_idx,
    geom_idx=geom_idx,
    cand_idx=cand_idx,
)


In [ ]:
# @title 3. Load model bank and define groups

bank_rows = []
bundles = []

for model_name in BANK_MODELS:
    for seed in BANK_SEEDS:
        try:
            bundle = load_rep_bundle(
                save_root=DEFAULT_SAVE_ROOT,
                run_group=DEFAULT_RUN_GROUP,
                dataset_name=BANK_DATASET,
                model_name=model_name,
                seed=seed,
                checkpoint_tag=CHECKPOINT_TAG,
            )
            if REP_LAYER not in bundle["layer_names"]:
                raise ValueError(f"{REP_LAYER} not found for {model_name} seed {seed}")

            bank_rows.append({
                "dataset": BANK_DATASET,
                "model_name": model_name,
                "seed": seed,
                "arch": model_name,
                "group": model_name,   # edit this later if you want another regime
                "run_dir": bundle["run_dir"],
                "step": bundle["step"],
            })
            bundles.append(bundle)

        except Exception as e:
            print(f"Skipping {model_name} seed {seed}: {e}")

bank_df = pd.DataFrame(bank_rows)
bank_csv = os.path.join(TABLE_DIR, "loaded_model_bank.csv")
bank_df.to_csv(bank_csv, index=False)

print(bank_df)
print(f"\nLoaded {len(bundles)} models.")

GROUP_COL = "group"
group_values = sorted(bank_df[GROUP_COL].unique().tolist())
if len(group_values) != 2:
    raise RuntimeError(
        f"This experiment currently expects exactly 2 groups. Found: {group_values}"
    )

print("Group values:", group_values)
print("If later you want standard-vs-adversarial or another regime, edit bank_df['group'] before Cell 5.")


In [ ]:
# @title 4. Generic model helpers

num_devices = jax.local_device_count()

def batch_iterator(X, batch_size=256):
    for i in range(0, len(X), batch_size):
        yield X[i:i+batch_size]

def pad_to_devices(x):
    n = x.shape[0]
    per_dev = int(np.ceil(n / num_devices))
    n_pad = per_dev * num_devices - n
    if n_pad > 0:
        x = np.concatenate([x, np.repeat(x[-1:], n_pad, axis=0)], axis=0)
    return x.reshape((num_devices, per_dev) + x.shape[1:]), n

@partial(jax.pmap, in_axes=(None, None, 0), static_broadcasted_argnums=(0,))
def compute_logits_chunk(apply_fn, variables, x_batch):
    return apply_fn(variables, x_batch, train=False)

@partial(jax.pmap, in_axes=(None, None, None, 0), static_broadcasted_argnums=(0, 1))
def compute_layer_chunk(apply_fn, layer_name, variables, x_batch):
    _, mut = apply_fn(variables, x_batch, train=False, mutable=["intermediates"])
    act = mut["intermediates"][layer_name][0]
    return act.reshape((act.shape[0], -1))

def batched_logits(bundle, X, batch_size=256):
    outs = []
    apply_fn = bundle["model"].apply
    variables = bundle["variables"]

    for xb in batch_iterator(X, batch_size=batch_size):
        x_sh, n_orig = pad_to_devices(xb)
        logits = compute_logits_chunk(apply_fn, variables, x_sh)
        logits = np.array(logits, dtype=np.float32).reshape(-1, np.array(logits).shape[-1])[:n_orig]
        outs.append(logits)

    return np.concatenate(outs, axis=0)

def extract_layer_features(bundle, X, layer_name=REP_LAYER, batch_size=256):
    outs = []
    apply_fn = bundle["model"].apply
    variables = bundle["variables"]

    for xb in batch_iterator(X, batch_size=batch_size):
        x_sh, n_orig = pad_to_devices(xb)
        feats = compute_layer_chunk(apply_fn, layer_name, variables, x_sh)
        feats = np.array(feats, dtype=np.float32).reshape(-1, np.array(feats).shape[-1])[:n_orig]
        outs.append(feats)

    return np.concatenate(outs, axis=0)

def true_class_margin(logits, y_true):
    N, C = logits.shape
    true_scores = logits[np.arange(N), y_true]
    mask = np.eye(C, dtype=bool)[y_true]
    other = np.where(mask, -np.inf, logits)
    other_max = np.max(other, axis=1)
    return true_scores - other_max

print("Helpers ready.")


In [ ]:
# @title 5. Candidate clean correctness and benchmark image selection

clean_logits_candidates_path = os.path.join(ARRAY_DIR, "clean_logits_candidates.npy")
clean_correct_candidates_path = os.path.join(ARRAY_DIR, "clean_correct_candidates.npy")

if os.path.exists(clean_logits_candidates_path) and os.path.exists(clean_correct_candidates_path):
    clean_logits_cand = np.load(clean_logits_candidates_path)
    clean_correct_cand = np.load(clean_correct_candidates_path)
else:
    M = len(bundles)
    N = len(X_cand)
    C = DATASET_CFG[BANK_DATASET]["num_classes"]

    clean_logits_cand = np.zeros((M, N, C), dtype=np.float32)
    clean_correct_cand = np.zeros((M, N), dtype=bool)

    for m_idx, bundle in enumerate(tqdm(bundles, desc="Clean candidate logits")):
        logits = batched_logits(bundle, X_cand, batch_size=LOGIT_BATCH_SIZE)
        clean_logits_cand[m_idx] = logits
        clean_correct_cand[m_idx] = (np.argmax(logits, axis=1) == y_cand)

    np.save(clean_logits_candidates_path, clean_logits_cand)
    np.save(clean_correct_candidates_path, clean_correct_cand)

# total and per-group counts
total_counts = clean_correct_cand.sum(axis=0)

group_counts = {}
for g in group_values:
    idx = bank_df.index[bank_df[GROUP_COL] == g].tolist()
    group_counts[g] = clean_correct_cand[idx].sum(axis=0)

selected = []
for i in range(len(X_cand)):
    ok_total = total_counts[i] >= MIN_CLEAN_CORRECT_TOTAL
    ok_groups = all(group_counts[g][i] >= MIN_CLEAN_CORRECT_PER_GROUP_ALLMODELS for g in group_values)
    if ok_total and ok_groups:
        selected.append(i)

selected = selected[:N_BENCH_IMAGES]

if len(selected) < N_BENCH_IMAGES:
    raise RuntimeError(
        f"Only {len(selected)} candidate images meet the selection criteria; "
        f"need {N_BENCH_IMAGES}."
    )

selected = np.array(selected, dtype=np.int64)
bench_idx = cand_idx[selected]

X_bench = X_cand[selected]
y_bench = y_cand[selected]

print("Selected benchmark images:", X_bench.shape)
print("Mean total correct count:", total_counts[selected].mean())
for g in group_values:
    print(f"Mean {g} correct count:", group_counts[g][selected].mean())

np.savez_compressed(
    os.path.join(ARRAY_DIR, "benchmark_image_selection.npz"),
    bench_idx=bench_idx,
    selected_indices_within_cand=selected,
    y_bench=y_bench,
)


In [ ]:
# @title 6. Boundary-safe smooth geometric flow bank

H = W = DATASET_CFG[BANK_DATASET]["image_size"]
yy, xx = np.meshgrid(np.arange(H, dtype=np.float32), np.arange(W, dtype=np.float32), indexing="ij")

sx = xx / (W - 1.0)
sy = yy / (H - 1.0)

def scalar_mode(u, v):
    return np.sin(np.pi * u * sx) * np.sin(np.pi * v * sy)

def build_broad_flow_bank():
    """
    Returns array [24, H, W, 2], each with RMS displacement = 1 pixel.
    """
    flows = []
    labels = []

    for (u, v) in MODE_LIST:
        phi = scalar_mode(u, v).astype(np.float32)

        fx = np.stack([phi, np.zeros_like(phi)], axis=-1)
        fy = np.stack([np.zeros_like(phi), phi], axis=-1)

        for f, name in [(fx, f"mode({u},{v})_dx"), (fy, f"mode({u},{v})_dy")]:
            rms = np.sqrt(np.mean(f[..., 0] ** 2 + f[..., 1] ** 2))
            f = f / (rms + 1e-12)
            flows.append(f.astype(np.float32))
            labels.append(name)

    flows = np.stack(flows, axis=0)   # [24,H,W,2]
    return flows, labels

FLOW_BANK, FLOW_LABELS = build_broad_flow_bank()
np.save(os.path.join(ARRAY_DIR, "broad_flow_bank.npy"), FLOW_BANK)

def warp_with_flow_pixel(x_pix, flow, coeff):
    """
    x_pix: [H,W,3] in [0,1]
    flow:  [H,W,2]
    coeff: scalar multiplier in pixels RMS
    """
    dx = coeff * flow[..., 0]
    dy = coeff * flow[..., 1]

    x_src = xx - dx
    y_src = yy - dy

    out = np.zeros_like(x_pix, dtype=np.float32)
    for c in range(3):
        out[..., c] = ndi.map_coordinates(
            x_pix[..., c],
            [y_src, x_src],
            order=1,
            mode="reflect",
        )

    return np.clip(out, 0.0, 1.0)

x0_pix = denormalize_to_pixel(X_famlearn[0], BANK_DATASET)

fig, axes = plt.subplots(2, 3, figsize=(8, 5))
axes = axes.ravel()

axes[0].imshow(x0_pix)
axes[0].set_title("Original")
axes[0].axis("off")

for k in range(1, 6):
    xw = warp_with_flow_pixel(x0_pix, FLOW_BANK[k-1], coeff=1.0)
    axes[k].imshow(xw)
    axes[k].set_title(FLOW_LABELS[k-1])
    axes[k].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# @title 7. Build base tangent banks

def build_base_tangent_bank(X_norm, flow_bank, fd_eps=0.5):
    """
    Returns [N, B, H, W, C] float16
    tangent_b(x) = (T_{+eps flow_b}(x) - T_{-eps flow_b}(x)) / (2 eps)
    """
    N = len(X_norm)
    B = flow_bank.shape[0]
    out = np.zeros((N, B, H, W, 3), dtype=np.float16)

    for n in tqdm(range(N), desc="Base tangent bank"):
        x_pix = denormalize_to_pixel(X_norm[n], BANK_DATASET)

        tangents = []
        for b in range(B):
            flow = flow_bank[b]

            xp = warp_with_flow_pixel(x_pix, flow, coeff=fd_eps)
            xm = warp_with_flow_pixel(x_pix, flow, coeff=-fd_eps)

            xp_n = normalize_from_pixel(xp, BANK_DATASET)
            xm_n = normalize_from_pixel(xm, BANK_DATASET)

            t = (xp_n - xm_n) / (2.0 * fd_eps)
            tangents.append(t.astype(np.float16))

        out[n] = np.stack(tangents, axis=0)

    return out

famlearn_tangent_path = os.path.join(ARRAY_DIR, "base_tangent_bank_famlearn.npy")
geom_tangent_path = os.path.join(ARRAY_DIR, "base_tangent_bank_geom.npy")
bench_tangent_path = os.path.join(ARRAY_DIR, "base_tangent_bank_bench.npy")

if not os.path.exists(famlearn_tangent_path):
    T_famlearn = build_base_tangent_bank(X_famlearn, FLOW_BANK, fd_eps=FD_EPS)
    np.save(famlearn_tangent_path, T_famlearn)
    del T_famlearn
    gc.collect()

if not os.path.exists(geom_tangent_path):
    T_geom = build_base_tangent_bank(X_geom, FLOW_BANK, fd_eps=FD_EPS)
    np.save(geom_tangent_path, T_geom)
    del T_geom
    gc.collect()

if not os.path.exists(bench_tangent_path):
    T_bench = build_base_tangent_bank(X_bench, FLOW_BANK, fd_eps=FD_EPS)
    np.save(bench_tangent_path, T_bench)
    del T_bench
    gc.collect()

print("Saved tangent banks.")

In [ ]:
# @title 8. Learn fixed family basis A and derived flow basis F_nat

family_basis_path = os.path.join(ARRAY_DIR, "learned_family_basis.npz")

def learn_family_from_tangent_covariance(base_tangent_path, k=16):
    """
    base_tangent_path: [N,24,H,W,C]
    Computes C = E_x[T_x T_x^T] where T_x is [24,d] of base tangents.
    Returns top-k eigvecs A: [24,k].
    """
    T_bank = np.load(base_tangent_path, mmap_mode="r")
    N = T_bank.shape[0]
    B = T_bank.shape[1]

    C = np.zeros((B, B), dtype=np.float64)

    for n in tqdm(range(N), desc="Family covariance"):
        Tn = np.array(T_bank[n], dtype=np.float32).reshape(B, -1)   # [24,d]
        C += Tn @ Tn.T

    C /= N

    evals, evecs = np.linalg.eigh(C)
    order = np.argsort(evals)[::-1]
    evals = evals[order]
    evecs = evecs[:, order]   # [24,24]

    A = evecs[:, :k].astype(np.float32)   # [24,k]
    return C.astype(np.float32), A, evals.astype(np.float32)

if os.path.exists(family_basis_path):
    dd = np.load(family_basis_path, allow_pickle=True)
    C_tan = dd["C_tan"]
    A = dd["A"]                     # [24,K]
    evals_C = dd["evals_C"]
    F_nat = dd["F_nat"]             # [K,H,W,2]
    print("Loaded cached family basis.")
else:
    C_tan, A, evals_C = learn_family_from_tangent_covariance(famlearn_tangent_path, k=K_FAMILY)
    F_nat = np.tensordot(A.T, FLOW_BANK, axes=(1, 0)).astype(np.float32)   # [K,H,W,2]

    np.savez_compressed(
        family_basis_path,
        C_tan=C_tan,
        A=A,
        evals_C=evals_C,
        F_nat=F_nat,
    )

print("A shape:", A.shape)
print("F_nat shape:", F_nat.shape)
print("Top family eigenvalues:", evals_C[:10])

# visualize first few learned family basis elements as actual smooth warps
x0_pix = denormalize_to_pixel(X_famlearn[0], BANK_DATASET)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for j in range(8):
    xw = warp_with_flow_pixel(x0_pix, F_nat[j], coeff=0.75)
    ax = axes.ravel()[j]
    ax.imshow(xw)
    ax.set_title(f"F_nat {j+1}")
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# @title 9. Build learned tangent banks and estimate class-conditional G^(y)

learned_geom_tangent_path = os.path.join(ARRAY_DIR, "learned_tangent_bank_geom.npy")
learned_bench_tangent_path = os.path.join(ARRAY_DIR, "learned_tangent_bank_bench.npy")

def combine_base_tangents_with_A(base_tangent_path, A, out_path):
    """
    base tangent bank: [N,24,H,W,C]
    A: [24,K]
    learned tangent bank output: [N,K,H,W,C] float16
    """
    T_bank = np.load(base_tangent_path, mmap_mode="r")
    N = T_bank.shape[0]
    K = A.shape[1]

    out = np.zeros((N, K, H, W, 3), dtype=np.float16)

    for n in tqdm(range(N), desc=f"Combining tangents -> {os.path.basename(out_path)}"):
        Tn = np.array(T_bank[n], dtype=np.float32)  # [24,H,W,C]
        Un = np.tensordot(Tn, A, axes=(0, 0))       # [H,W,C,K]
        Un = np.transpose(Un, (3, 0, 1, 2))         # [K,H,W,C]
        out[n] = Un.astype(np.float16)

    np.save(out_path, out)

if not os.path.exists(learned_geom_tangent_path):
    combine_base_tangents_with_A(geom_tangent_path, A, learned_geom_tangent_path)

if not os.path.exists(learned_bench_tangent_path):
    combine_base_tangents_with_A(bench_tangent_path, A, learned_bench_tangent_path)

@partial(jax.pmap, in_axes=(None, None, None, 0, 0), static_broadcasted_argnums=(0, 1))
def compute_H_chunk(apply_fn, layer_name, variables, x_batch, tangents_batch):
    """
    x_batch: [per_device_batch,H,W,C]
    tangents_batch: [per_device_batch,K,H,W,C]
    returns H per image: [per_device_batch,K,K]
    """
    def target_fn(x):
        _, mut = apply_fn(variables, x[None, ...], train=False, mutable=["intermediates"])
        act = mut["intermediates"][layer_name][0]
        return act.reshape(-1)

    def H_one(img, tangents):
        def single_jvp(t):
            _, tangent = jax.jvp(target_fn, (img,), (t,))
            return tangent
        V = jax.vmap(single_jvp)(tangents)   # [K,D]
        return V @ V.T

    return jax.vmap(H_one)(x_batch, tangents_batch)

def estimate_H_bank(bundle, X, learned_tangent_path, batch_size_total=32):
    U_bank = np.load(learned_tangent_path, mmap_mode="r")
    N = len(X)
    out = []

    apply_fn = bundle["model"].apply
    variables = bundle["variables"]

    for start in range(0, N, batch_size_total):
        stop = min(start + batch_size_total, N)
        xb = X[start:stop]
        ub = np.array(U_bank[start:stop], dtype=np.float32)

        x_sh, n_orig = pad_to_devices(xb)
        u_sh, _ = pad_to_devices(ub)

        H_chunk = compute_H_chunk(apply_fn, REP_LAYER, variables, x_sh, u_sh)
        H_chunk = np.array(H_chunk, dtype=np.float32).reshape(-1, K_FAMILY, K_FAMILY)[:n_orig]
        out.append(H_chunk)

    return np.concatenate(out, axis=0)

G_class_global_path = os.path.join(ARRAY_DIR, "G_class_global.npy")
H_local_bench_path = os.path.join(ARRAY_DIR, "H_local_bench.npy")

if os.path.exists(G_class_global_path) and os.path.exists(H_local_bench_path):
    G_class_global = np.load(G_class_global_path)   # [M,C,K,K]
    H_local_bench = np.load(H_local_bench_path)     # [M,N_bench,K,K]
    print("Loaded cached G_class_global and H_local_bench.")
else:
    M = len(bundles)
    G_class_global = np.zeros((M, N_CLASSES, K_FAMILY, K_FAMILY), dtype=np.float32)
    H_local_bench = np.zeros((M, len(X_bench), K_FAMILY, K_FAMILY), dtype=np.float32)

    class_counts_geom = np.array([(y_geom == c).sum() for c in range(N_CLASSES)], dtype=np.int64)
    if np.any(class_counts_geom == 0):
        raise RuntimeError("At least one class has zero examples in geometry split.")

    for m_idx, bundle in enumerate(tqdm(bundles, desc="Estimating H/G objects")):
        H_geom = estimate_H_bank(bundle, X_geom, learned_geom_tangent_path, batch_size_total=GRAM_BATCH_SIZE_TOTAL)
        H_bench = estimate_H_bank(bundle, X_bench, learned_bench_tangent_path, batch_size_total=GRAM_BATCH_SIZE_TOTAL)

        H_local_bench[m_idx] = H_bench

        for c in range(N_CLASSES):
            G_class_global[m_idx, c] = H_geom[y_geom == c].mean(axis=0)

        del H_geom, H_bench
        gc.collect()

    np.save(G_class_global_path, G_class_global)
    np.save(H_local_bench_path, H_local_bench)

print("G_class_global shape:", G_class_global.shape)
print("H_local_bench shape:", H_local_bench.shape)


In [ ]:
# @title 10. Clean correctness and true-class margins on benchmark images

clean_bench_path = os.path.join(ARRAY_DIR, "clean_bench_logits_margins.npy.npz")

if os.path.exists(clean_bench_path):
    cc = np.load(clean_bench_path, allow_pickle=True)
    clean_logits = cc["clean_logits"]        # [M,N,C]
    clean_correct = cc["clean_correct"]      # [M,N]
    clean_margins = cc["clean_margins"]      # [M,N]
    print("Loaded cached clean benchmark logits.")
else:
    M = len(bundles)
    N = len(X_bench)
    C = DATASET_CFG[BANK_DATASET]["num_classes"]

    clean_logits = np.zeros((M, N, C), dtype=np.float32)
    clean_correct = np.zeros((M, N), dtype=bool)
    clean_margins = np.zeros((M, N), dtype=np.float32)

    for m_idx, bundle in enumerate(tqdm(bundles, desc="Clean benchmark logits")):
        logits = batched_logits(bundle, X_bench, batch_size=LOGIT_BATCH_SIZE)
        clean_logits[m_idx] = logits
        clean_correct[m_idx] = (np.argmax(logits, axis=1) == y_bench)
        clean_margins[m_idx] = true_class_margin(logits, y_bench)

    np.savez_compressed(
        clean_bench_path,
        clean_logits=clean_logits,
        clean_correct=clean_correct,
        clean_margins=clean_margins,
    )

print("Mean clean-correct rate on benchmark images:", clean_correct.mean())


In [ ]:
# @title 11. Balanced discovery/test splits

split_rows = []

group_to_idx = {
    g: bank_df.index[bank_df[GROUP_COL] == g].tolist()
    for g in group_values
}

for g in group_values:
    if len(group_to_idx[g]) != (DISC_PER_GROUP + TEST_PER_GROUP):
        raise RuntimeError(
            f"Group {g} has {len(group_to_idx[g])} models, "
            f"expected {DISC_PER_GROUP + TEST_PER_GROUP}."
        )

for s in range(N_SPLITS):
    rng_split = np.random.default_rng(GLOBAL_SEED + 1000 + s)
    split_spec = {"split_id": s}

    discovery = {}
    heldout = {}

    for g in group_values:
        disc = sorted(rng_split.choice(group_to_idx[g], size=DISC_PER_GROUP, replace=False).tolist())
        tst = sorted([i for i in group_to_idx[g] if i not in disc])

        discovery[g] = disc
        heldout[g] = tst

    split_spec["discovery"] = discovery
    split_spec["heldout"] = heldout
    split_rows.append(split_spec)

splits_json = os.path.join(TABLE_DIR, "balanced_model_splits.json")
with open(splits_json, "w") as f:
    json.dump(split_rows, f, indent=2)

print("Saved:", splits_json)
print(split_rows[0])


In [ ]:
# @title 12. Probe derivation helpers

def derive_contrast_probes(deltaG, probes_per_side=2):
    """
    deltaG = G_A - G_B
    Returns dict:
      pos_dirs [P,K], neg_dirs [P,K]
    """
    evals, evecs = np.linalg.eigh(deltaG)
    order = np.argsort(evals)[::-1]
    evals = evals[order]
    evecs = evecs[:, order]

    pos_dirs = evecs[:, :probes_per_side].T.astype(np.float32)
    neg_dirs = evecs[:, -probes_per_side:].T.astype(np.float32)

    return {
        "pos_dirs": pos_dirs,
        "neg_dirs": neg_dirs,
        "evals": evals.astype(np.float32),
    }

def derive_random_contrast_probes(deltaG, n_random=64, probes_per_side=2, seed=0):
    rng_local = np.random.default_rng(seed)
    Z = rng_local.standard_normal((n_random, K_FAMILY)).astype(np.float32)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12

    scores = np.einsum("nk,kl,nl->n", Z, deltaG, Z)

    pos_idx = np.argsort(scores)[::-1][:probes_per_side]
    neg_idx = np.argsort(scores)[:probes_per_side]

    return {
        "pos_dirs": Z[pos_idx].astype(np.float32),
        "neg_dirs": Z[neg_idx].astype(np.float32),
        "scores": scores.astype(np.float32),
    }

def derive_pooled_top_probes(Gbar, deltaG, top_pool=8, probes_per_side=2):
    evals, evecs = np.linalg.eigh(Gbar)
    order = np.argsort(evals)[::-1]
    evecs = evecs[:, order]

    Z = evecs[:, :top_pool].T.astype(np.float32)  # [top_pool,K]
    scores = np.einsum("nk,kl,nl->n", Z, deltaG, Z)

    pos_idx = np.argsort(scores)[::-1][:probes_per_side]
    neg_idx = np.argsort(scores)[:probes_per_side]

    return {
        "pos_dirs": Z[pos_idx].astype(np.float32),
        "neg_dirs": Z[neg_idx].astype(np.float32),
        "scores": scores.astype(np.float32),
    }

def derive_label_permutation_probes(G_disc_by_group, probes_per_side=2, seed=0):
    """
    G_disc_by_group: dict group -> [n_disc,C,K,K]
    permutes discovery labels while preserving counts
    """
    rng_local = np.random.default_rng(seed)

    G_all = []
    labels = []
    for g, arr in G_disc_by_group.items():
        for i in range(arr.shape[0]):
            G_all.append(arr[i])
            labels.append(g)

    G_all = np.stack(G_all, axis=0)   # [n_disc_total,C,K,K]
    labels = np.array(labels)

    perm = rng_local.permutation(len(labels))
    labels_perm = labels[perm]

    # assume exactly 2 groups
    gA, gB = group_values

    return G_all, labels_perm, gA, gB

def combine_family_dir_to_flow(z):
    """
    z: [K]
    returns normalized combined flow [H,W,2] with RMS displacement = 1 pixel
    """
    flow = np.tensordot(z, F_nat, axes=(0, 0)).astype(np.float32)  # [H,W,2]
    rms = np.sqrt(np.mean(flow[..., 0] ** 2 + flow[..., 1] ** 2))
    flow = flow / (rms + 1e-12)
    return flow

def probe_dict_to_flows(probe_dict):
    pos_flows = np.stack([combine_family_dir_to_flow(z) for z in probe_dict["pos_dirs"]], axis=0)
    neg_flows = np.stack([combine_family_dir_to_flow(z) for z in probe_dict["neg_dirs"]], axis=0)
    return {
        "pos_flows": pos_flows.astype(np.float32),
        "neg_flows": neg_flows.astype(np.float32),
    }


In [ ]:
# @title 13. Main experiment: class-conditional held-out diagnostic probes

probe_meta_csv = os.path.join(TABLE_DIR, "probe_sets.csv")
results_csv = os.path.join(TABLE_DIR, "heldout_diagnostic_probe_results.csv")
image_sep_csv = os.path.join(TABLE_DIR, "heldout_image_level_separation.csv")
model_score_csv = os.path.join(TABLE_DIR, "heldout_model_scores.csv")

def build_transformed_stack_for_image(x_norm, pos_flows, neg_flows):
    """
    pos_flows: [P,H,W,2], neg_flows: [P,H,W,2]
    returns transformed images:
      [2P, 2 signs, A amps, H, W, C]
    """
    x_pix = denormalize_to_pixel(x_norm, BANK_DATASET)

    probe_flows = np.concatenate([pos_flows, neg_flows], axis=0)
    n_probes = probe_flows.shape[0]

    out = np.zeros((n_probes, 2, len(FLOW_RMS_AMPS), H, W, 3), dtype=np.float32)
    signs = [+1.0, -1.0]

    for p_idx in range(n_probes):
        flow = probe_flows[p_idx]
        for s_idx, sgn in enumerate(signs):
            for a_idx, amp in enumerate(FLOW_RMS_AMPS):
                xt = warp_with_flow_pixel(x_pix, flow, coeff=sgn * amp)
                out[p_idx, s_idx, a_idx] = normalize_from_pixel(xt, BANK_DATASET)

    return out

def model_image_regime_score(bundle, transformed_stack, y_true, clean_margin):
    """
    transformed_stack: [2P, 2, A, H, W, C]
    returns:
      regime_score,
      per_probe_responses [2P]
    """
    n_probes = transformed_stack.shape[0]
    flat = transformed_stack.reshape(n_probes * 2 * len(FLOW_RMS_AMPS), H, W, 3)

    logits = batched_logits(bundle, flat, batch_size=LOGIT_BATCH_SIZE)
    logits = logits.reshape(n_probes, 2, len(FLOW_RMS_AMPS), -1)

    probe_resp = np.zeros(n_probes, dtype=np.float32)

    for p_idx in range(n_probes):
        vals = []
        for s_idx in range(2):
            for a_idx in range(len(FLOW_RMS_AMPS)):
                lt = logits[p_idx, s_idx, a_idx]   # [C]
                true_score = lt[y_true]
                other = np.delete(lt, y_true)
                other_max = np.max(other)
                margin = true_score - other_max
                vals.append(clean_margin - margin)
        probe_resp[p_idx] = np.mean(vals)

    P = n_probes // 2
    score = probe_resp[:P].mean() - probe_resp[P:].mean()
    return float(score), probe_resp

result_rows = []
image_sep_rows = []
model_score_rows = []
probe_meta_rows = []

gA, gB = group_values

for split_spec in tqdm(split_rows, desc="Discovery/test splits"):
    split_id = split_spec["split_id"]
    disc_A = split_spec["discovery"][gA]
    disc_B = split_spec["discovery"][gB]
    test_A = split_spec["heldout"][gA]
    test_B = split_spec["heldout"][gB]
    test_all = test_A + test_B

    # discovery tensors by group: [n_disc, C, K, K]
    G_disc_A = G_class_global[disc_A]
    G_disc_B = G_class_global[disc_B]

    # label-permutation helper pool
    G_disc_by_group = {gA: G_disc_A, gB: G_disc_B}

    for c in range(N_CLASSES):
        img_idx_class = np.where(y_bench == c)[0]
        if len(img_idx_class) == 0:
            continue

        # group means for this class
        G_A_c = G_disc_A[:, c].mean(axis=0)
        G_B_c = G_disc_B[:, c].mean(axis=0)
        deltaG_c = G_A_c - G_B_c
        Gbar_c = np.concatenate([G_disc_A[:, c], G_disc_B[:, c]], axis=0).mean(axis=0)

        # label permutation for this class
        G_all_perm, labels_perm, _, _ = derive_label_permutation_probes(
            G_disc_by_group,
            probes_per_side=PROBES_PER_SIDE,
            seed=GLOBAL_SEED + 10000 + split_id + 37 * c,
        )
        G_A_perm = G_all_perm[labels_perm == gA, c].mean(axis=0)
        G_B_perm = G_all_perm[labels_perm == gB, c].mean(axis=0)
        deltaG_perm = G_A_perm - G_B_perm

        probe_sets = {
            "contrast": derive_contrast_probes(deltaG_c, probes_per_side=PROBES_PER_SIDE),
            "random_contrast": derive_random_contrast_probes(
                deltaG_c,
                n_random=N_RANDOM_CAND,
                probes_per_side=PROBES_PER_SIDE,
                seed=GLOBAL_SEED + 20000 + split_id + 53 * c,
            ),
            "pooled_sensitivity": derive_pooled_top_probes(
                Gbar_c,
                deltaG_c,
                top_pool=TOP_POOLED_EIGS,
                probes_per_side=PROBES_PER_SIDE,
            ),
            "label_permutation": derive_contrast_probes(
                deltaG_perm,
                probes_per_side=PROBES_PER_SIDE,
            ),
        }

        for probe_type, probe_dict in probe_sets.items():
            flows = probe_dict_to_flows(probe_dict)
            pos_flows = flows["pos_flows"]
            neg_flows = flows["neg_flows"]

            for side_name, dirs in [("pos", probe_dict["pos_dirs"]), ("neg", probe_dict["neg_dirs"])]:
                for i, z in enumerate(dirs):
                    probe_meta_rows.append({
                        "split_id": split_id,
                        "class_idx": c,
                        "probe_type": probe_type,
                        "side": side_name,
                        "probe_idx": i,
                        "z_norm": float(np.linalg.norm(z)),
                    })

            per_model_image_score = {}

            for img_idx in img_idx_class:
                transformed_stack = build_transformed_stack_for_image(
                    X_bench[img_idx],
                    pos_flows,
                    neg_flows,
                )

                for model_idx in test_all:
                    if not clean_correct[model_idx, img_idx]:
                        continue

                    bundle = bundles[model_idx]
                    score, probe_resp = model_image_regime_score(
                        bundle,
                        transformed_stack,
                        y_bench[img_idx],
                        clean_margins[model_idx, img_idx],
                    )

                    per_model_image_score[(model_idx, img_idx)] = score

                    model_score_rows.append({
                        "split_id": split_id,
                        "class_idx": c,
                        "probe_type": probe_type,
                        "model_idx": model_idx,
                        "model_name": bank_df.iloc[model_idx]["model_name"],
                        "seed": int(bank_df.iloc[model_idx]["seed"]),
                        "group": bank_df.iloc[model_idx][GROUP_COL],
                        "img_idx": img_idx,
                        "score": score,
                    })

                del transformed_stack
                gc.collect()

            # image-level held-out separation
            img_sep_vals = []
            for img_idx in img_idx_class:
                scores_A = [
                    per_model_image_score[(m, img_idx)]
                    for m in test_A
                    if (m, img_idx) in per_model_image_score
                ]
                scores_B = [
                    per_model_image_score[(m, img_idx)]
                    for m in test_B
                    if (m, img_idx) in per_model_image_score
                ]

                if len(scores_A) < MIN_GROUP_MODELS_PER_IMAGE or len(scores_B) < MIN_GROUP_MODELS_PER_IMAGE:
                    continue

                sep = float(np.mean(scores_A) - np.mean(scores_B))
                img_sep_vals.append(sep)

                image_sep_rows.append({
                    "split_id": split_id,
                    "class_idx": c,
                    "probe_type": probe_type,
                    "img_idx": img_idx,
                    "image_sep": sep,
                    "n_groupA_models": len(scores_A),
                    "n_groupB_models": len(scores_B),
                })

            split_mean_sep = np.mean(img_sep_vals) if len(img_sep_vals) > 0 else np.nan

            # optional held-out model AUC
            model_means = []
            model_labels = []

            for m in test_all:
                vals = [
                    per_model_image_score[(m, img_idx)]
                    for img_idx in img_idx_class
                    if (m, img_idx) in per_model_image_score
                ]
                if len(vals) == 0:
                    continue
                model_means.append(np.mean(vals))
                model_labels.append(1 if bank_df.iloc[m][GROUP_COL] == gA else 0)

            auc = np.nan
            if len(np.unique(model_labels)) == 2 and len(model_means) >= 2:
                auc = roc_auc_score(model_labels, model_means)

            result_rows.append({
                "split_id": split_id,
                "class_idx": c,
                "probe_type": probe_type,
                "mean_image_separation": split_mean_sep,
                "model_auc": auc,
                "n_valid_images": len(img_sep_vals),
                "n_test_models": len(test_all),
            })

probe_meta_df = pd.DataFrame(probe_meta_rows)
probe_meta_df.to_csv(probe_meta_csv, index=False)

results_df = pd.DataFrame(result_rows)
results_df.to_csv(results_csv, index=False)

image_sep_df = pd.DataFrame(image_sep_rows)
image_sep_df.to_csv(image_sep_csv, index=False)

model_score_df = pd.DataFrame(model_score_rows)
model_score_df.to_csv(model_score_csv, index=False)

print("Saved:")
print(" ", probe_meta_csv)
print(" ", results_csv)
print(" ", image_sep_csv)
print(" ", model_score_csv)

display(results_df.head())


In [ ]:
# @title 14. Summary, bootstrap, and main figures

results_df = pd.read_csv(results_csv)
image_sep_df = pd.read_csv(image_sep_csv)
model_score_df = pd.read_csv(model_score_csv)

summary_csv = os.path.join(TABLE_DIR, "summary_results.csv")
boot_csv = os.path.join(TABLE_DIR, "bootstrap_results.csv")
summary_ci_csv = os.path.join(TABLE_DIR, "summary_with_ci.csv")

# point summaries aggregated over splits and classes
summary_rows = []
for probe_type in ["contrast", "random_contrast", "pooled_sensitivity", "label_permutation"]:
    sub = results_df[results_df["probe_type"] == probe_type]
    summary_rows.append({
        "probe_type": probe_type,
        "mean_image_separation": sub["mean_image_separation"].mean(),
        "mean_model_auc": sub["model_auc"].mean(),
        "n_split_class_rows": len(sub),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(summary_csv, index=False)

# bootstrap over images within each split/class/probe_type, then average over split/class rows
rng_boot = np.random.default_rng(GLOBAL_SEED + 999)
boot_rows = []

for b in tqdm(range(N_BOOT), desc="Bootstrap over images"):
    for probe_type in ["contrast", "random_contrast", "pooled_sensitivity", "label_permutation"]:
        sep_vals = []
        auc_vals = []

        sub_probe = image_sep_df[image_sep_df["probe_type"] == probe_type]
        keys = sorted(set(zip(sub_probe["split_id"], sub_probe["class_idx"])))

        for split_id, class_idx in keys:
            sub = sub_probe[(sub_probe["split_id"] == split_id) & (sub_probe["class_idx"] == class_idx)]
            vals = sub["image_sep"].values
            if len(vals) == 0:
                continue

            samp = rng_boot.choice(vals, size=len(vals), replace=True)
            sep_vals.append(np.mean(samp))

            # model AUC bootstrap (appendix only)
            sub_scores = model_score_df[
                (model_score_df["split_id"] == split_id) &
                (model_score_df["class_idx"] == class_idx) &
                (model_score_df["probe_type"] == probe_type)
            ]

            if len(sub_scores) > 0:
                img_ids = sub_scores["img_idx"].unique()
                samp_img = rng_boot.choice(img_ids, size=len(img_ids), replace=True)

                model_means = []
                model_labels = []

                for model_idx, g in sub_scores.groupby("model_idx"):
                    vals_model = []
                    for img_id in samp_img:
                        rows_img = g[g["img_idx"] == img_id]
                        if len(rows_img) > 0:
                            vals_model.extend(rows_img["score"].tolist())

                    if len(vals_model) > 0:
                        model_means.append(np.mean(vals_model))
                        model_labels.append(1 if g["group"].iloc[0] == gA else 0)

                if len(np.unique(model_labels)) == 2 and len(model_means) >= 2:
                    auc_vals.append(roc_auc_score(model_labels, model_means))

        boot_rows.append({
            "boot": b,
            "probe_type": probe_type,
            "quantity": "mean_image_separation",
            "value": np.nanmean(sep_vals),
        })
        boot_rows.append({
            "boot": b,
            "probe_type": probe_type,
            "quantity": "mean_model_auc",
            "value": np.nanmean(auc_vals),
        })

boot_df = pd.DataFrame(boot_rows)
boot_df.to_csv(boot_csv, index=False)

summary_ci_rows = []
for probe_type in ["contrast", "random_contrast", "pooled_sensitivity", "label_permutation"]:
    for quantity in ["mean_image_separation", "mean_model_auc"]:
        vals = boot_df[
            (boot_df["probe_type"] == probe_type) &
            (boot_df["quantity"] == quantity)
        ]["value"].values

        summary_ci_rows.append({
            "probe_type": probe_type,
            "quantity": quantity,
            "mean": np.mean(vals),
            "ci_low": np.percentile(vals, 2.5),
            "ci_high": np.percentile(vals, 97.5),
        })

summary_ci_df = pd.DataFrame(summary_ci_rows)
summary_ci_df.to_csv(summary_ci_csv, index=False)

display(summary_df)
display(summary_ci_df)

def save_figure_both(fig, stem):
    pdf_path = os.path.join(FIG_DIR, f"{stem}.pdf")
    svg_path = os.path.join(FIG_DIR, f"{stem}.svg")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(svg_path, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", pdf_path)
    print("Saved:", svg_path)

order = ["contrast", "random_contrast", "pooled_sensitivity", "label_permutation"]

# Main figure: held-out image separation
fig, ax = plt.subplots(figsize=(8.5, 5.0))
sub = summary_ci_df[summary_ci_df["quantity"] == "mean_image_separation"].set_index("probe_type").loc[order]

means = sub["mean"].values
yerr = np.vstack([
    np.maximum(means - sub["ci_low"].values, 0.0),
    np.maximum(sub["ci_high"].values - means, 0.0),
])

ax.bar(np.arange(len(order)), means, yerr=yerr, capsize=4)
ax.axhline(0.0, linestyle="--", color="black", linewidth=1.2)
ax.set_xticks(np.arange(len(order)))
ax.set_xticklabels(order, rotation=20, ha="right")
ax.set_ylabel(f"Held-out image separation\n({gA} score - {gB} score)")
ax.set_title("Class-conditional diagnostic probes from $G_{f,P}$")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure_both(fig, "heldout_image_separation")

# Appendix-ish: held-out model AUC
fig, ax = plt.subplots(figsize=(8.5, 5.0))
sub = summary_ci_df[summary_ci_df["quantity"] == "mean_model_auc"].set_index("probe_type").loc[order]

means = sub["mean"].values
yerr = np.vstack([
    np.maximum(means - sub["ci_low"].values, 0.0),
    np.maximum(sub["ci_high"].values - means, 0.0),
])

ax.bar(np.arange(len(order)), means, yerr=yerr, capsize=4)
ax.axhline(0.5, linestyle="--", color="black", linewidth=1.2)
ax.set_xticks(np.arange(len(order)))
ax.set_xticklabels(order, rotation=20, ha="right")
ax.set_ylabel("Held-out model AUC")
ax.set_title("Appendix: held-out architecture/regime AUC")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure_both(fig, "heldout_model_auc")

# Per-class heatmap for main metric
heat_rows = []
for probe_type in order:
    for c in range(N_CLASSES):
        sub = results_df[(results_df["probe_type"] == probe_type) & (results_df["class_idx"] == c)]
        heat_rows.append({
            "probe_type": probe_type,
            "class_idx": c,
            "mean_sep": sub["mean_image_separation"].mean(),
        })

heat_df = pd.DataFrame(heat_rows)
heat_mat = heat_df.pivot(index="probe_type", columns="class_idx", values="mean_sep").loc[order]

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(heat_mat, cmap="viridis", annot=True, fmt=".2f", ax=ax)
ax.set_title("Per-class held-out image separation")
ax.set_xlabel("Class")
ax.set_ylabel("")
fig.tight_layout()
save_figure_both(fig, "per_class_image_separation_heatmap")


In [ ]:
# @title 15. Qualitative probe figure

results_df = pd.read_csv(results_csv)

# choose best split/class for contrast probes
sub_best = results_df[results_df["probe_type"] == "contrast"].sort_values("mean_image_separation", ascending=False).iloc[0]
best_split = int(sub_best["split_id"])
best_class = int(sub_best["class_idx"])

split_spec = [s for s in split_rows if s["split_id"] == best_split][0]
disc_A = split_spec["discovery"][gA]
disc_B = split_spec["discovery"][gB]

G_A_c = G_class_global[disc_A][:, best_class].mean(axis=0)
G_B_c = G_class_global[disc_B][:, best_class].mean(axis=0)
deltaG_c = G_A_c - G_B_c
probe_dict = derive_contrast_probes(deltaG_c, probes_per_side=PROBES_PER_SIDE)
flows = probe_dict_to_flows(probe_dict)

# choose a benchmark image of that class
img_candidates = np.where(y_bench == best_class)[0]
img_idx = int(img_candidates[0])

x0_pix = denormalize_to_pixel(X_bench[img_idx], BANK_DATASET)

fig, axes = plt.subplots(3, 3, figsize=(8, 8))

axes[0, 0].imshow(x0_pix)
axes[0, 0].set_title(f"Original (class {best_class})")
axes[0, 0].axis("off")

for i in range(PROBES_PER_SIDE):
    axes[0, i+1].imshow(warp_with_flow_pixel(x0_pix, flows["pos_flows"][i], coeff=FLOW_RMS_AMPS[-1]))
    axes[0, i+1].set_title(f"{gA}-favoring {i+1}")
    axes[0, i+1].axis("off")

axes[1, 0].imshow(x0_pix)
axes[1, 0].set_title("Original")
axes[1, 0].axis("off")

for i in range(PROBES_PER_SIDE):
    axes[1, i+1].imshow(warp_with_flow_pixel(x0_pix, flows["neg_flows"][i], coeff=FLOW_RMS_AMPS[-1]))
    axes[1, i+1].set_title(f"{gB}-favoring {i+1}")
    axes[1, i+1].axis("off")

axes[2, 0].imshow(x0_pix)
axes[2, 0].set_title("Original")
axes[2, 0].axis("off")

for i in range(PROBES_PER_SIDE):
    axes[2, i+1].imshow(warp_with_flow_pixel(x0_pix, flows["pos_flows"][i], coeff=-FLOW_RMS_AMPS[-1]))
    axes[2, i+1].set_title(f"{gA}-fav {i+1} (-)")
    axes[2, i+1].axis("off")

plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "qualitative_class_conditional_probes.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(FIG_DIR, "qualitative_class_conditional_probes.svg"), bbox_inches="tight")
plt.show()
